In [37]:
!pip install pytorch-fid torchmetrics -q
!pip install -U gradio huggingface_hub diffusers
!pip install git+https://github.com/sberbank-ai/Real-ESRGAN.git

  Cloning https://github.com/sberbank-ai/Real-ESRGAN.git to /tmp/pip-req-build-v_sga1_l
  Running command git clone --filter=blob:none --quiet https://github.com/sberbank-ai/Real-ESRGAN.git /tmp/pip-req-build-v_sga1_l
  Resolved https://github.com/sberbank-ai/Real-ESRGAN.git to commit 362a0316878f41dbdfbb23657b450c3353de5acf
  Preparing metadata (setup.py) ... done


In [38]:
!git clone -b Frontend https://github.com/youlkar/damage-diffusion.git

fatal: destination path 'damage-diffusion' already exists and is not an empty directory.


In [39]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

# Authenticate with Google account
auth.authenticate_user()
service = build('drive', 'v3')

FILE_ID = '1raocuWSZzWU9M3whQM2idltFBig9JnsU'
OUTPUT  = '/content/final_model.pt'
print("Starting download via Drive API...", flush=True)
request    = service.files().get_media(fileId=FILE_ID)
fh         = io.FileIO(OUTPUT, 'wb')
downloader = MediaIoBaseDownload(fh, request, chunksize=100 * 1024 * 1024)  # 100MB chunks

done = False
while not done:
    status, done = downloader.next_chunk()
    pct = int(status.progress() * 100)
    print(f'\rDownloading... {pct}%', end='', flush=True)

fh.close()
print(f'\nDone! Size: {os.path.getsize(OUTPUT) / 1e9:.2f} GB', flush=True)
CKPT_PATH = OUTPUT
print(f'Use this path: {CKPT_PATH}', flush=True)

Starting download via Drive API...


Downloading... 100%
Done! Size: 0.00 GB
Use this path: /content/final_model.pt


In [40]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os

auth.authenticate_user()
service = build('drive', 'v3')

FILE_ID = '1v2uGKKTtm9LHdb1uRkfWqPxJ3oG2i5xy'
OUTPUT = '/content/checkpoint_metrics.pt'

print("Starting download via Drive API...", flush=True)
request = service.files().get_media(fileId=FILE_ID)
fh = io.FileIO(OUTPUT, 'wb')
downloader = MediaIoBaseDownload(fh, request, chunksize=100 * 1024 * 1024)
done = False
while not done:
    status, done = downloader.next_chunk()
    pct = int(status.progress() * 100)
    print(f'\rDownloading... {pct}%', end='', flush=True)
fh.close()
print(f'\nDone! Size: {os.path.getsize(OUTPUT)/1e6:.2f} MB', flush=True)
print(f'Use this path: {OUTPUT}', flush=True)

Starting download via Drive API...


Downloading... 100%
Done! Size: 0.00 MB
Use this path: /content/checkpoint_metrics.pt


In [41]:
import torch

metrics = torch.load('/content/checkpoint_metrics.pt')
print("Keys:", list(metrics.keys()))
for k, v in metrics.items():
    if isinstance(v, list):
        print(f"{k}: {len(v)} values, first 5: {v[:5]}")
    else:
        print(f"{k}: {v}")

Keys: ['train_loss', 'val_loss', 'fid_score', 'kid_score', 'mask_sensitivity_score', 'learning_rate', 'epoch']
train_loss: 11 values, first 5: [0.024445165151751838, 0.022418164109356355, 0.02394903761193936, 0.024054641829976856, 0.02394014940770386]
val_loss: 11 values, first 5: [0.016334901005029677, 0.015730063850060105, 0.015435038786381482, 0.016080100554972887, 0.01858007926493883]
fid_score: 2 values, first 5: [65.28135681152344, 68.39143371582031]
kid_score: 2 values, first 5: [0.023685377091169357, 0.022836998105049133]
mask_sensitivity_score: 2 values, first 5: [141.40145874023438, 141.25161743164062]
learning_rate: 11 values, first 5: [2.1921202840320077e-06, 1.8767810889299085e-06, 1.609427140540686e-06, 1.390322284933351e-06, 1.2196827521475405e-06]
epoch: 11 values, first 5: [89, 90, 91, 92, 93]


In [42]:
import sys, os
# Removed sys.path.insert and os.chdir from this cell to avoid conflicts.
os.system('git checkout Frontend')

32768

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import sys, os, importlib
sys.path.insert(0, '/content/damage-diffusion/backend')
spec = importlib.util.find_spec('inference')
print('find_spec:', spec)
if spec:
    print('origin:', spec.origin)

find_spec: ModuleSpec(name='inference', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7d865983e2d0>, origin='/content/damage-diffusion/backend/inference.py')
origin: /content/damage-diffusion/backend/inference.py


In [45]:
import gradio as gr
import torch
import numpy as np
from PIL import Image
from PIL import ImageFilter, ImageEnhance
import torchvision.transforms as transforms
import sys, os
import matplotlib.pyplot as plt

import huggingface_hub
if not hasattr(huggingface_hub, 'cached_download'):
    huggingface_hub.cached_download = lambda *args, **kwargs: None

from RealESRGAN import RealESRGAN

sys.path.insert(0, '/content/damage-diffusion/backend')
from inference import load_model, generate_from_mask
from utils.metrics import compute_iou, compute_pixel_accuracy
from utils.visualization import denormalize
import os
import urllib.request

# globals
model = None
device = None

# Manually create the folder and download the weights using standard Python
os.makedirs('weights', exist_ok=True)
weights_path = 'weights/RealESRGAN_x4.pth'

if not os.path.exists(weights_path):
    print("Manually downloading Real-ESRGAN weights...")
    urllib.request.urlretrieve("https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth", weights_path)
    print("Download complete!")

# Load the upscaler
upscaler = RealESRGAN(device, scale=4)
upscaler.load_weights(weights_path, download=False)

def load(checkpoint_path, device_choice):
    global model, device
    if device_choice == "cuda" and not torch.cuda.is_available():
        device_choice = "cpu"
    device = device_choice
    if not os.path.exists(checkpoint_path):
        return f" Checkpoint not found at {checkpoint_path}"
    try:
        model, scheduler = load_model(checkpoint_path, device=device)
        return f" Model loaded from {checkpoint_path} on {device}"
    except Exception as e:
        return f" ERROR loading model: {str(e)}"



def upscale_image(pil_img):
    """Combine AI concrete textures with mathematically preserved cracks"""
    pil_img = pil_img.convert('RGB')

    with torch.no_grad():
        ai_img = upscaler.predict(pil_img)

    # The Traditional Version: Math-based upscale that perfectly preserves the original cracks
    new_size = (pil_img.width * 4, pil_img.height * 4)
    trad_img = pil_img.resize(new_size, Image.Resampling.LANCZOS)

    # Create a mask to separate the crack from the concrete
    gray_np = np.array(trad_img.convert('L'))

    # Find pixels significantly darker than the concrete average
    mask_np = np.where(gray_np < (gray_np.mean() - 35), 255, 0).astype(np.uint8)
    crack_mask = Image.fromarray(mask_np)

    # Blur the mask slightly so the sharp cracks blend seamlessly
    crack_mask = crack_mask.filter(ImageFilter.GaussianBlur(1.5))

    # Paste the perfect cracks onto the enhanced background
    final_img = Image.composite(trad_img, ai_img, crack_mask)

    return final_img

def preprocess_mask(mask_img, image_size=128):
    """Handle both Sketchpad dict and regular image inputs"""
    is_sketch = False

    # Extract image from Sketchpad dict if needed
    if isinstance(mask_img, dict):
        is_sketch = True
        # Sketchpad returns {'background': array, 'layers': [arrays], 'composite': array}
        if mask_img.get('composite') is not None:
            mask_img = mask_img['composite']
        elif mask_img.get('background') is not None:
            mask_img = mask_img['background']
        elif mask_img.get('layers') and len(mask_img['layers']) > 0:
            mask_img = mask_img['layers'][0]

    # Ensure it's a numpy array
    if not isinstance(mask_img, np.ndarray):
        return None

    # Convert to PIL
    pil = Image.fromarray(mask_img.astype(np.uint8)).convert('L')
    pil = pil.resize((image_size, image_size))

    # Convert to tensor
    t = transforms.ToTensor()(pil)

    # Binarize based on the source
    if is_sketch:
        # Sketchpad is black-on-white. Invert so black pen (<0.5) becomes 1 (crack).
        t = (t < 0.5).float()
    else:
        # Uploads are usually white-on-black. Keep white (>0.5) as 1 (crack).
        t = (t > 0.5).float()

    return t.unsqueeze(0)

def tensor_to_pil(t):
    """Convert tensor to PIL image"""
    t = t.squeeze(0).cpu()
    t = denormalize(t).clamp(0, 1)
    return transforms.ToPILImage()(t)

# ============= GENERATE TAB =============
def generate(upload, draw, num_steps, num_samples):
    """Generate crack image from mask (upload or draw)"""
    if model is None:
        return None, " Load a model checkpoint first."

    # Prioritize upload, fallback to draw
    mask_np = upload if upload is not None else draw

    # Check if mask exists (safe for numpy arrays)
    if mask_np is None or (isinstance(mask_np, np.ndarray) and mask_np.size == 0):
        return None, " Please upload or draw a mask first."

    try:
        mask_tensor = preprocess_mask(mask_np)
        if mask_tensor is None:
            return None, " Invalid mask format."

        generated = generate_from_mask(
            model, mask_tensor,
            num_samples=num_samples,
            num_inference_steps=num_steps,
            device=device
        )

        # output_images = [upscale_image(tensor_to_pil(img)) for img in generated]
        output_images = [tensor_to_pil(img) for img in generated]

        return output_images, " Generation and upscaling complete!"

    except Exception as e:
        return None, f" Generation error: {str(e)}"

# ============= EVALUATE TAB =============
def evaluate(gen_img, gt_mask):
    """Compute IoU and Pixel Accuracy"""
    if gen_img is None or gt_mask is None:
        return " Please provide both generated image and ground truth mask."

    try:
        # FIX: Immediately convert to 'L' (Grayscale) to destroy any hidden Alpha channels from PNG uploads
        gen_pil = Image.fromarray(gen_img).convert('L').resize((128, 128))
        gen_t = transforms.ToTensor()(gen_pil).unsqueeze(0)

        mask_t = preprocess_mask(gt_mask)

        # Calculate the average color of this specific concrete image
        img_mean = gen_t.mean()

        # Flag pixels as a crack ONLY if they are significantly darker than the concrete average
        gen_binary = (gen_t < (img_mean - 0.25)).float()

        iou = compute_iou(gen_binary, mask_t)
        acc = compute_pixel_accuracy(gen_binary, mask_t)

        return f" Metrics Computed:\n\nIoU: {iou:.4f}\nPixel Accuracy: {acc:.4f}"

    except Exception as e:
        return f" Evaluation error: {str(e)}"

# ============= METRICS DASHBOARD TAB =============
def plot_metrics():
    """Load metrics.pt and plot training curves"""
    try:
        if not os.path.exists('/content/checkpoint_metrics.pt'):
            return None, " checkpoint_metrics.pt not found at /content/checkpoint_metrics.pt"

        metrics = torch.load('/content/checkpoint_metrics.pt')

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        fig.suptitle('Training Metrics', fontsize=16)

        # Plot 1: Loss
        ax = axes[0, 0]
        ax.plot(metrics['epoch'], metrics['train_loss'], label='Train Loss', marker='o')
        ax.plot(metrics['epoch'], metrics['val_loss'], label='Val Loss', marker='s')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Loss over Epochs')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Plot 2: FID Score
        ax = axes[0, 1]
        fid_epochs = [metrics['epoch'][i*10] for i in range(len(metrics['fid_score']))]
        ax.plot(fid_epochs, metrics['fid_score'], label='FID Score', marker='o', color='orange')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('FID Score')
        ax.set_title('FID Score over Epochs')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Plot 3: KID Score
        ax = axes[1, 0]
        ax.plot(fid_epochs, metrics['kid_score'], label='KID Score', marker='s', color='green')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('KID Score')
        ax.set_title('KID Score over Epochs')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Plot 4: Learning Rate
        ax = axes[1, 1]
        ax.plot(metrics['epoch'], metrics['learning_rate'], label='Learning Rate', marker='^', color='red')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Learning Rate')
        ax.set_title('Learning Rate Schedule')
        ax.legend()
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        return fig, " Metrics loaded and plotted successfully!"
    except Exception as e:
        return None, f" Metrics error: {str(e)}"


# ============= GRADIO UI =============
with gr.Blocks(title="DamageDiffusion") as demo:
    gr.Markdown("DamageDiffusion -- Crack Image Generator")
    gr.Markdown("Mask-conditioned diffusion model for synthetic concrete crack generation")

    # Model Control Section
    with gr.Row():
        ckpt_input = gr.Textbox(
            label="Checkpoint Path",
            value="/content/drive/MyDrive/DamageDiffusion/checkpoints_full/final_model.pt",
            placeholder="/content/drive/MyDrive/DamageDiffusion/checkpoints_full/final_model.pt"
        )
        device_drop = gr.Dropdown(
            choices=["cuda", "cpu"],
            value="cuda",
            label="Device"
        )
        load_btn = gr.Button("Load Model", variant="primary")

    load_status = gr.Textbox(label="Status", interactive=False, lines=2)
    load_btn.click(load, inputs=[ckpt_input, device_drop], outputs=load_status)

    # Tabs for different functionalities
    with gr.Tabs():
        # ===== GENERATE TAB =====
        with gr.Tab(" Generate"):
            gr.Markdown("**Upload a mask or draw one below**, then click Generate.")

            with gr.Row():
                with gr.Column():
                    mask_upload = gr.Image(
                        label="Option 1: Upload Mask",
                        type="numpy",
                        sources=["upload"]
                    )
                    gr.Markdown("**-- OR --**")
                    mask_draw = gr.Sketchpad(
                        label="Option 2: Draw Mask",
                        type="numpy",
                        image_mode="RGB",
                        canvas_size=(512, 512)
                    )

                    num_steps = gr.Slider(
                        10, 250, value=50, step=10,
                        label="Inference Steps (more = better quality but slower)"
                    )
                    num_samples = gr.Slider(
                        1, 8, value=1, step=1,
                        label="Number of Samples"
                    )
                    gen_btn = gr.Button("Generate", variant="primary", size="lg")

                with gr.Column():
                    gen_output = gr.Gallery(label="Generated Images", columns=2, format="png")
                    gen_status = gr.Textbox(label="Status", interactive=False, lines=3)

            gen_btn.click(
                generate,
                inputs=[mask_upload, mask_draw, num_steps, num_samples],
                outputs=[gen_output, gen_status]
            )

        # ===== EVALUATE TAB =====
        with gr.Tab(" Evaluate"):
            gr.Markdown("**Upload a generated image and ground truth mask** to compute metrics.")

            with gr.Row():
                eval_gen = gr.Image(
                    label="Generated Image",
                    type="numpy",
                    sources=["upload"]
                )
                eval_mask = gr.Image(
                    label="Ground Truth Mask",
                    type="numpy",
                    sources=["upload"]
                )

            eval_btn = gr.Button("Compute Metrics", variant="primary", size="lg")
            eval_output = gr.Textbox(label="Metrics", interactive=False, lines=6)

            eval_btn.click(
                evaluate,
                inputs=[eval_gen, eval_mask],
                outputs=eval_output
            )

        # ===== METRICS DASHBOARD TAB =====
        with gr.Tab(" Training Metrics"):
            gr.Markdown("**Training curves from logs_full/metrics.pt**")

            metrics_btn = gr.Button("Load Metrics", variant="primary", size="lg")
            metrics_plot = gr.Plot(label="Metrics Visualization")
            metrics_status = gr.Textbox(label="Status", interactive=False, lines=2)

            metrics_btn.click(
                plot_metrics,
                outputs=[metrics_plot, metrics_status]
            )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2783a8306a12b1b513.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loaded training config:
Model channels: (128, 256, 512, 512)
Timesteps: 1000
Image size: 128
Loaded model from /content/drive/MyDrive/DamageDiffusion/checkpoints_full/final_model.pt
Trained for 99 epochs
Generating 8 samples with 220 steps...
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2783a8306a12b1b513.gradio.live


In [46]:
import inspect
print(inspect.getsource(generate))

def generate(upload, draw, num_steps, num_samples):
    """Generate crack image from mask (upload or draw)"""
    if model is None:
        return None, " Load a model checkpoint first."

    # Prioritize upload, fallback to draw
    mask_np = upload if upload is not None else draw

    # Check if mask exists (safe for numpy arrays)
    if mask_np is None or (isinstance(mask_np, np.ndarray) and mask_np.size == 0):
        return None, " Please upload or draw a mask first."

    try:
        mask_tensor = preprocess_mask(mask_np)
        if mask_tensor is None:
            return None, " Invalid mask format."

        generated = generate_from_mask(
            model, mask_tensor, 
            num_samples=num_samples, 
            num_inference_steps=num_steps, 
            device=device
        )
        
        # output_images = [upscale_image(tensor_to_pil(img)) for img in generated]
        output_images = [tensor_to_pil(img) for img in generated]

        return output_images, " G

In [47]:
import traceback
print(traceback.format_exc())

NoneType: None

